# Piecewise model reconstruction

In a directed model:

- If a component $C_1$ is unreachable from another component $C_2$, then $C_1$ may be fit independently of $C_2$.
- A component can be fit conditional only on the arcs _into_ it.

In [ ]:
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression

from climate_attitudes.visualisation import configure_mpl
from ising import Ising, SymmetricIsing

RANDOM_SEED = 202604241124

configure_mpl(Path("../fonts"))

## Initialise model

Create a model with three spins, in two components. The first components has a bidirectional edge between nodes $1$ and $2$. The second component contains a single node, with an arc from $1$.

In [ ]:
model = SymmetricIsing(
    coupling=np.array(
        [
            [0.0, 1.0],
            [0.0, 0.0],
        ]
    ),
    infer_structure=True,
    # upper_triangular=True,
    rng=RANDOM_SEED,
)

model = Ising(
    coupling=np.array(
        [
            [0.0, 1.0],
            [0.0, 0.0],
        ]
    ),
    infer_structure=True,
    upper_triangular=True,
    rng=RANDOM_SEED,
)

X = model.sample(n_samples=100_000)

In [ ]:
fig, ax = model.draw(figsize=(2, 1.2))

fig.savefig("2spins.png", bbox_inches="tight")

## Fit $C_1$ parameters

We first fit $h_1$ and $J_{2,1}$ by solving:

$$S^t_1 \sim \operatorname{Logistic}(h_1 + J_{2,1}\cdot S^{t-1}_2)$$

In [ ]:
X

In [ ]:
(X[:, 0] == X[:, 1]).sum() / 10000

In [ ]:
res = LogisticRegression().fit(X[:, 1][:, None], X[:, 0])

In [ ]:
h_1 = res.intercept_ / 2
J_21 = res.coef_[0] / 2

In [ ]:
J_21

Then $h_2$ and $J_{1,2}$ by solving:

$$S^t_2 \sim \operatorname{Logistic}(h_2 + J_{1,2}\cdot S^{t}_1)$$

In [ ]:
res = LogisticRegression().fit(X[:, 0][:, None], X[:, 1])
h_2 = res.intercept_ / 2
J_12 = res.coef_[0] / 2

In [ ]:
J_12

## Fit $C_2$ parameters



In [ ]:
1 / (1 + np.exp(2 * -1 * (0.38 * 1)))

In [ ]:
import scipy as sp

In [ ]:
1 / 2 * (1 / 2 + sp.special.expit(2))

In [ ]:
sp.special.expit(2)

In [ ]:
prop_1_given_1 = (np.array([1, 1]) == X).all(axis=1).sum() / (X[:, 1] == 1).sum()
prop_1_given_neg1 = (np.array([1, -1]) == X).all(axis=1).sum() / (X[:, 1] == -1).sum()

In [ ]:
prob_1_given_1 = sp.special.expit(2 * 0.416)
prob_1_given_neg1 = sp.special.expit(2 * -0.416)

In [ ]:
prop_1_given_1, prob_1_given_1

In [ ]:
prop_1_given_neg1, prob_1_given_neg1

In [ ]:
x_11_prop = (np.array([1, 1]) == X).all(axis=1).sum() / 100_000
x_neg11_prop = (np.array([-1, 1]) == X).all(axis=1).sum() / 100_000
x_11_to_neg11_prop = (
    (X[:-1] == np.array([1, 1])).all(axis=1) & (X[1:] == np.array([-1, 1])).all(axis=1)
).sum() / 100_000
x_neg11_to_11_prop = (
    (X[:-1] == np.array([-1, 1])).all(axis=1) & (X[1:] == np.array([1, 1])).all(axis=1)
).sum() / 100_000

In [ ]:
x_11_prop * x_11_to_neg11_prop

In [ ]:
x_neg11_prop * x_neg11_to_11_prop

In [ ]:
x_11_prop, x_11_to_neg11_prop

In [ ]:
x_neg11_prop, x_neg11_to_11_prop